### Mithil Agneya
### 4/29/26
### Cleaned Project Milestone #3

#### Importing libraries and dataset

In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv('Electric_Vehicle_Population_Data.csv')

df.isna().sum()

VIN (1-10)                                             0
County                                                12
City                                                  12
State                                                  0
Postal Code                                           12
Model Year                                             0
Make                                                   0
Model                                                  0
Electric Vehicle Type                                  0
Clean Alternative Fuel Vehicle (CAFV) Eligibility      0
Electric Range                                        12
Legislative District                                 703
DOL Vehicle ID                                         0
Vehicle Location                                      20
Electric Utility                                      12
2020 Census Tract                                     12
dtype: int64

In [52]:
df.columns

Index(['VIN (1-10)', 'County', 'City', 'State', 'Postal Code', 'Model Year',
       'Make', 'Model', 'Electric Vehicle Type',
       'Clean Alternative Fuel Vehicle (CAFV) Eligibility', 'Electric Range',
       'Legislative District', 'DOL Vehicle ID', 'Vehicle Location',
       'Electric Utility', '2020 Census Tract'],
      dtype='object')

#### To make the dataframe easier to work with in Python, I renamed all the columns to match snake_case. This removes capital letters, spaces, parentheses, and dashes, preventing syntax errors when referencing columns later.

In [53]:
df.columns = [col.lower().replace(' ','_').replace('(','').replace(')','').replace('-','_') for col in df.columns]
df.columns

Index(['vin_1_10', 'county', 'city', 'state', 'postal_code', 'model_year',
       'make', 'model', 'electric_vehicle_type',
       'clean_alternative_fuel_vehicle_cafv_eligibility', 'electric_range',
       'legislative_district', 'dol_vehicle_id', 'vehicle_location',
       'electric_utility', '2020_census_tract'],
      dtype='object')

#### Several of my exploratory questions rely heavily on geographic location. It is essentially impossible to accurately guess or impute a vehicle's county/city without external identifying data. Because the number of missing geographical rows is incredibly low compared to the total dataset size, dropping them ensures high data integrity without sacrificing statistical power.

In [54]:
df = df.dropna(subset=['county', 'city'])

#### Remapping vehicle types. Allows for clear categorization

In [55]:
df['vehicle_type'] = df['electric_vehicle_type'].map({
    'Battery Electric Vehicle (BEV)': 'EV',
    'Plug-in Hybrid Electric Vehicle (PHEV)': 'Hybrid'
})
df['vehicle_type']

0         EV
1         EV
2         EV
3         EV
4         EV
          ..
280828    EV
280829    EV
280830    EV
280831    EV
280832    EV
Name: vehicle_type, Length: 280821, dtype: object

#### For this step, I first converted 0 values to NaN because a 0-mile range was a data entry error that had to be treated as missing data. Instead of dropping all these rows or using an inaccurate global average, I filled the NaNs with the median range of each specific vehicle make and model. This was much more precise since identical car models share the same battery specs. Finally, I dropped the few remaining NaNs that belonged to rare models without enough data to establish a median, ensuring my final analysis wasn't skewed.

In [56]:
df['electric_range_cleaned'] = df['electric_range'].replace(0, np.nan)
df['electric_range_cleaned'] = df['electric_range_cleaned'].fillna(
    df.groupby(['make', 'model'])['electric_range_cleaned'].transform('median')
)
df['electric_range_cleaned']
df.isna().sum()

vin_1_10                                               0
county                                                 0
city                                                   0
state                                                  0
postal_code                                            0
model_year                                             0
make                                                   0
model                                                  0
electric_vehicle_type                                  0
clean_alternative_fuel_vehicle_cafv_eligibility        0
electric_range                                        12
legislative_district                                 691
dol_vehicle_id                                         0
vehicle_location                                       8
electric_utility                                       0
2020_census_tract                                      0
vehicle_type                                           0
electric_range_cleaned         

In [58]:
df = df.dropna(subset=['electric_range_cleaned'])
df['electric_range_cleaned'].isna().sum()
df['electric_range_cleaned']

0          75.0
1         308.0
2         215.0
4         291.0
5          72.0
          ...  
280826     38.0
280827    291.0
280829    220.0
280831    238.0
280832    291.0
Name: electric_range_cleaned, Length: 203676, dtype: float64

#### Making age column to be able to differentiate between cars from multiple years

In [59]:
df['age'] = 2025 - df['model_year']

#### Sorting counties into area types, for suburban and urban. Chose top 10 as a safe metric

In [62]:
top_10_counties = df['county'].value_counts().head(10).index
df['area_type'] = df['county'].apply(lambda x: 'Urban' if x in top_10_counties else 'Rural')

In [65]:
keep = ['make', 'model', 'model_year', 'city', 'county', 'area_type', 'vehicle_type', 'electric_range_cleaned', 'age']
df_cleaned = df[keep]

In [66]:
df_cleaned

,make,model,model_year,city,county,area_type,vehicle_type,electric_range_cleaned,age
0,NISSAN,LEAF,2013,Kirkland,King,Urban,EV,75.0,12
1,TESLA,MODEL 3,2020,Bainbridge Island,Kitsap,Urban,EV,308.0,5
2,TESLA,MODEL 3,2018,Seattle,King,Urban,EV,215.0,7
4,TESLA,MODEL Y,2020,Kent,King,Urban,EV,291.0,5
5,BMW,I3,2015,Mukilteo,Snohomish,Urban,Hybrid,72.0,10
...,...,...,...,...,...,...,...,...,...
280826,CHEVROLET,VOLT,2013,Augusta,Richmond,Rural,Hybrid,38.0,12
280827,TESLA,MODEL Y,2023,Seattle,King,Urban,EV,291.0,2
280829,TESLA,MODEL 3,2022,Sultan,Snohomish,Urban,EV,220.0,3
280831,TESLA,MODEL X,2026,Mukilteo,Snohomish,Urban,EV,238.0,-1


#### Creating a new CSV File with cleaned data

In [67]:
df_cleaned.to_csv('Cleaned_EV_Data.csv', index=False)